# 01 — Exploration des données (EDA)

**Projet** : Estimation du prix de vente immobilier (AVM) avec XGBoost
**Cas métier** : Prédire le prix de vente net vendeur d'un bien résidentiel à partir de sa fiche descriptive, et publier une **fourchette** assortie d'un niveau de confiance, plutôt qu'un prix unique présenté comme exact.
**Jeu de données** : `real_estate_prices` — Biens immobiliers résidentiels et prix de vente

> Population de biens résidentiels mis en annonce sur une métropole française, avec leurs caractéristiques physiques, énergétiques, géographiques et administratives, et le prix de vente net vendeur observé. Les données sont générées par un modèle multiplicatif en log-prix : chaque caractéristique contribue par un facteur explicite (prime de quartier, effet surface en log, malus énergétique, effet étage), puis un bruit log-normal irréductible (sigma = 7 %) est ajouté. Le signal est réaliste — hétéroscédastique, non linéaire, avec colinéarités — et plafonne la performance atteignable.

## Objectifs pédagogiques

1. Charger un jeu de données tabulaire et en établir le profil (types, manquants, doublons).
1. Lire une distribution : détecter déséquilibre, outliers et colinéarité **avant** de modéliser.
1. Relier chaque observation statistique à une conséquence métier ou de modélisation.
1. Produire les figures qui serviront de référence dans les notebooks suivants.

**Objectifs transverses du dépôt**

- Comprendre le gradient boosting : arbres séquentiels qui corrigent les résidus, shrinkage et régularisation.
- Utiliser l'early stopping natif (`eval_set` + `early_stopping_rounds`) sans réinventer la boucle d'entraînement.
- Choisir un objectif de régression (`reg:squarederror`, `reg:absoluteerror`) selon la métrique pilotée.

## 0. Environnement

Toute la configuration vient de **Hydra** (`conf/`) : aucune valeur métier n'est codée en dur
dans ce notebook. Si `data/raw` est vide, le générateur synthétique du projet prend le relais
(voir `make data`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402
from loguru import logger  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
logger.remove()
logger.add(sys.stderr, level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (1500 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 1500

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 90000.0)")
print(f"Cible             : {CONFIG.data.target}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

**Pourquoi ce bloc d'initialisation**

- `CONFIG` est l'objet **Pydantic** validé : une clé incohérente échoue ici, pas en production.
- Les notebooks travaillent sur un échantillon réduit pour rester rapides ; `make train` utilise `data.n_samples` complet.
- `NB_PATHS` isole les écritures du notebook dans `outputs/notebooks`.

## 1. Chargement et premier contact

On ne regarde jamais un dataset sans vérifier trois choses : sa **forme** (lignes x colonnes),
ses **types** (un numérique lu comme texte casse tout) et ses **premières lignes** (les valeurs
ont-elles du sens métier ?).

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

**Ce qu'il faut retenir**

- Le contrat `real_estate_prices` est documenté dans `data/README.md` : chaque colonne y a une signification métier.
- Les identifiants et horodatages ne sont **pas** des features : ils servent à tracer et à splitter.

In [ ]:
from src.data.schemas import describe_schema, validation_report

# Types déclarés (contrat Pandera) vs types réellement lus : toute divergence est un signal.
contract = describe_schema("raw")
observed = pd.DataFrame({"dtype_lu": {str(k): str(v) for k, v in raw.dtypes.items()}})
contract.join(observed)[["dtype", "dtype_lu", "nullable", "unique", "checks"]]

**Ce qu'il faut retenir**

- La colonne `dtype` vient du **contrat**, `dtype_lu` de la source : elles doivent correspondre.
- Les `checks` (bornes, valeurs autorisées) sont la mémoire des règles métier — ils seront testés au notebook 02.

In [ ]:
report = validation_report(raw)
summary = pd.DataFrame(
    {
        "indicateur": [
            "lignes",
            "colonnes",
            "cellules manquantes",
            "taux de manquants",
            "mémoire (Ko)",
        ],
        "valeur": [
            report["n_rows"],
            report["n_columns"],
            report["missing_cells"],
            f"{report['missing_rate']:.2%}",
            round(report["memory_kb"], 1),
        ],
    }
)
summary

**Ce qu'il faut retenir**

- Un taux de manquants global faible peut cacher une colonne très incomplète : regarder **par colonne**.
- La mémoire indique si le dataset tient en RAM (sinon : pyarrow, chunking ou échantillonnage).

## 2. Valeurs manquantes

Où, combien, et surtout : **manquant au hasard ou pas** ? Un manquant informatif (ex. score de satisfaction non renseigné par les clients mécontents) est un signal, pas seulement un problème technique.

In [ ]:
missing = raw.isna().sum()
missing_frame = (
    pd.DataFrame({"manquants": missing, "taux": (missing / len(raw)).round(4)})
    .loc[lambda frame: frame["manquants"] > 0]
    .sort_values("manquants", ascending=False)
)
missing_frame

In [ ]:
if missing_frame.empty:
    print("Aucune valeur manquante dans cet échantillon.")
else:
    fig, axis = plt.subplots(figsize=(7.5, 0.55 * len(missing_frame) + 1.6))
    axis.barh(missing_frame.index[::-1], missing_frame["taux"][::-1] * 100, color="#d1495b")
    axis.set_xlabel("Cellules manquantes (%)")
    axis.set_title("Valeurs manquantes par colonne")
    fig.tight_layout()
    plt.show()

**Ce qu'il faut retenir**

- L'imputation doit être **apprise sur le train** (moyenne/médiane/constante) puis appliquée aux autres splits.
- Ajouter un indicateur binaire « valeur manquante » est souvent rentable quand le manquant est informatif.
- Notes du générateur : Valeurs manquantes volontaires sur `condition_score` (~6 %) et `condo_fees_eur` (~4 %) pour exercer l'imputation.; Outliers légitimes (~1 %) : biens d'exception (grande surface, centre historique, état haut de gamme)..

## 3. Distributions numériques

In [ ]:
numeric_columns = [column for column in raw.columns if pd.api.types.is_numeric_dtype(raw[column])]
numeric_columns = [column for column in numeric_columns if column != CONFIG.data.target]

n_plots = len(numeric_columns)
n_cols = 3
n_rows = int(np.ceil(n_plots / n_cols)) if n_plots else 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 2.7 * n_rows))
for axis, column in zip(np.atleast_1d(axes).ravel(), numeric_columns, strict=False):
    raw[column].hist(bins=30, ax=axis, color="#005f73", edgecolor="white")
    axis.set_title(column, fontsize=9)
    axis.tick_params(labelsize=7)
for axis in np.atleast_1d(axes).ravel()[len(numeric_columns) :]:
    axis.axis("off")
fig.suptitle("Distributions des variables numériques", y=1.005)
fig.tight_layout()
plt.show()

In [ ]:
raw[numeric_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(2)

**Ce qu'il faut retenir**

- Une distribution très asymétrique (max ≫ p99) justifie un **winsorising** ou un `log1p` plutôt qu'une suppression d'outliers.
- Des échelles hétérogènes (euros, Go, unités) imposent un **scaling** pour les modèles sensibles à la distance (SVM, k-NN, réseaux).
- Comparer `mean` et `50%` : un écart important signale une queue lourde.

## 4. Variables catégorielles

In [ ]:
categorical_columns = [
    column
    for column in raw.columns
    if not pd.api.types.is_numeric_dtype(raw[column])
    and column not in [*CONFIG.data.drop_columns, str(CONFIG.data.target)]
]

for column in categorical_columns:
    counts = raw[column].astype(str).value_counts()
    print(f"--- {column} ({len(counts)} modalités) ---")
    print((counts / len(raw)).map("{:.1%}".format).to_string())

**Ce qu'il faut retenir**

- Une modalité ultra-rare (< 1 %) doit être regroupée dans un bucket `rare` : sinon l'encodage one-hot crée des colonnes quasi vides et instables.
- Une cardinalité élevée (identifiants, codes postaux) appelle un **target encoding** régularisé plutôt qu'un one-hot.

## 5. La cible

Une cible continue ne se lit pas comme une étiquette : ce qui compte ici est la **forme** de la
distribution (asymétrie, queue lourde), l'**échelle** (les erreurs absolues croissent avec la
valeur) et la **stabilité** dans le temps. Ces trois lectures décident de la transformation de
cible, de la métrique de pilotage et de la largeur de fourchette à publier.

In [ ]:
target = CONFIG.data.target
values = raw[target].astype("float64")

summary = pd.DataFrame(
    {
        "statistique": [
            "effectif",
            "min",
            "P5",
            "médiane",
            "moyenne",
            "P95",
            "max",
            "écart-type",
            "asymétrie",
            "aplatissement",
        ],
        "valeur": [
            len(values),
            values.min(),
            values.quantile(0.05),
            values.median(),
            values.mean(),
            values.quantile(0.95),
            values.max(),
            values.std(),
            values.skew(),
            values.kurt(),
        ],
    }
)

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.4))
axes[0].hist(values, bins=50, color="#0a9396", edgecolor="white")
axes[0].axvline(values.median(), color="#d1495b", linestyle="--", label="médiane")
axes[0].axvline(values.mean(), color="#ee9b00", linestyle=":", label="moyenne")
axes[0].set_title(f"Distribution de `{target}`")
axes[0].set_xlabel(target)
axes[0].set_ylabel("nombre de biens")
axes[0].legend(fontsize=8)
axes[1].hist(values.clip(upper=values.quantile(0.99)), bins=50, color="#005f73", edgecolor="white")
axes[1].set_title("Zoom : 99 % de la population")
axes[1].set_xlabel(target)
fig.tight_layout()
plt.show()

summary.round(1)

**Ce qu'il faut retenir**

- Moyenne ≫ médiane = distribution **asymétrique à droite** : quelques biens d'exception tirent la moyenne. La RMSE hérite de cette sensibilité, pas la MAE.
- Une cible asymétrique se modélise mieux en **log** (`log1p` ou cible transformée) : la variance devient homogène et l'erreur relative devient l'erreur absolue.
- Ne **jamais** supprimer les valeurs extrêmes légitimes : les winsoriser (0,5-99,5 %) ou les traiter par une métrique robuste (erreur médiane) est préférable.

In [ ]:
# Asymétrie : la cible devient-elle gaussienne une fois passée en log ?
target = CONFIG.data.target
values = raw[target].astype("float64")
logged = np.log(values.clip(lower=1.0))

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.4))
for axis, series, title in (
    (axes[0], values, "échelle brute"),
    (axes[1], logged, "échelle logarithmique"),
):
    axis.hist(series, bins=50, color="#94d2bd", edgecolor="white")
    axis.set_title(f"Asymétrie = {series.skew():.2f} — {title}")
    axis.set_xlabel(target)
fig.tight_layout()
plt.show()

print(f"asymétrie brute : {values.skew():.3f}")
print(f"asymétrie log   : {logged.skew():.3f}")

**Ce qu'il faut retenir**

- Une asymétrie qui passe de > 1 à ≈ 0 en log confirme un processus **multiplicatif** : les facteurs de prix se cumulent en pourcentages, pas en euros.
- Conséquence pratique : entraîner sur le log-prix, puis revenir en euros avec une correction de biais (`exp(mu + sigma²/2)`), sinon on sous-estime systématiquement.
- Le générateur de ce projet implémente exactement ce mécanisme — l'observer ici, c'est vérifier que la donnée ressemble au métier.

In [ ]:
# Prix unitaire : la même lecture, ramenée à la surface (le ratio que le métier manipule).
target = CONFIG.data.target
unit_column = next(
    (column for column in ("surface_m2", "surface", "area_m2") if column in raw.columns), None
)
if unit_column is None:
    print("Aucune colonne de surface : le prix unitaire n'est pas calculable ici.")
else:
    unit_price = raw[target] / raw[unit_column].replace(0, np.nan)
    print(f"prix unitaire médian : {unit_price.median():,.0f} / {unit_column}")
    p10 = unit_price.quantile(0.10)
    p90 = unit_price.quantile(0.90)
    print(f"P10 / P90            : {p10:,.0f} / {p90:,.0f}")

    by_surface = (
        raw.assign(_bucket=pd.qcut(raw[unit_column], q=5, duplicates="drop"))
        .groupby("_bucket", observed=True)
        .agg(
            n=(target, "size"),
            prix_medien=(target, "median"),
            surface_mediane=(unit_column, "median"),
        )
    )
    by_surface["prix_m2_medien"] = by_surface["prix_medien"] / by_surface["surface_mediane"]
    display(by_surface.round(1))

**Ce qu'il faut retenir**

- Si le prix au m² **décroît** avec la surface, la relation prix/surface est **concave** : un modèle linéaire sur-évaluera les grands biens et sous-évaluera les studios.
- Ce ratio est aussi l'unité de lecture du métier : le présenter dans le rapport rend l'estimation crédible.
- Une feature dérivée (`surface_per_room`, `tax_per_surface`) est déjà déclarée dans `conf/preprocessing/default.yaml` : la configuration, pas le code, porte ces choix.

In [ ]:
# Signal par variable catégorielle : médiane de la cible par modalité (et volume associé).
target = CONFIG.data.target
rows = []
for column in categorical_columns:
    grouped = raw.groupby(raw[column].astype(str))[target].agg(["median", "mean", "count"])
    if grouped.empty or grouped["median"].nunique() < 2:
        continue
    rows.append(
        {
            "variable": column,
            "modalités": len(grouped),
            "médiane_min": round(float(grouped["median"].min()), 0),
            "médiane_max": round(float(grouped["median"].max()), 0),
            "ratio_max_min": round(
                float(grouped["median"].max() / max(grouped["median"].min(), 1.0)), 2
            ),
            "modalité_la_plus_chère": grouped["median"].idxmax(),
        }
    )
signal_frame = pd.DataFrame(rows).sort_values("ratio_max_min", ascending=False)
signal_frame

**Ce qu'il faut retenir**

- Un `ratio_max_min` élevé (≥ 1,5) entre modalités = un facteur de prix puissant, directement exploitable par le modèle.
- Un ratio proche de 1 ne signifie pas « inutile » : la variable peut interagir avec une autre (étage × ascenseur).
- Le volume par modalité compte autant que l'écart : un segment à 30 observations ne justifie pas une feature dédiée.

In [ ]:
# Signal par variable numérique : médiane de la cible par quartile.
target = CONFIG.data.target
plots = [column for column in numeric_columns if raw[column].nunique() > 4][:6]
fig, axes = plt.subplots(2, 3, figsize=(11.0, 5.4))
for axis, column in zip(np.atleast_1d(axes).ravel(), plots, strict=False):
    buckets = pd.qcut(raw[column], q=4, duplicates="drop")
    grouped = raw.groupby(buckets, observed=True)[target].median()
    grouped.plot.bar(ax=axis, color="#005f73", edgecolor="white")
    axis.set_title(f"Médiane de la cible par quartile — {column}", fontsize=8.5)
    axis.set_ylabel("")
    axis.tick_params(labelsize=7, axis="x", rotation=20)
for axis in np.atleast_1d(axes).ravel()[len(plots) :]:
    axis.axis("off")
fig.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Une relation **monotone** est apprise facilement, y compris par un modèle linéaire.
- Une relation en **U** ou **concave** impose des transformations (binning, log, splines) ou un modèle non linéaire : c'est précisément le cas de la surface et de l'année de construction.
- Le binning par quantiles est aussi une recette déclarée dans `conf/preprocessing/default.yaml` (`bin`) : l'EDA et la configuration racontent la même histoire.

## 6. Colinéarité et structure

In [ ]:
correlation = raw[numeric_columns].corr(numeric_only=True)
fig, axis = plt.subplots(figsize=(6.6, 5.4))
image = axis.imshow(correlation.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
axis.set_xticks(
    range(len(correlation.columns)), correlation.columns, rotation=45, ha="right", fontsize=7
)
axis.set_yticks(range(len(correlation.index)), correlation.index, fontsize=7)
for row in range(correlation.shape[0]):
    for column in range(correlation.shape[1]):
        value = correlation.iloc[row, column]
        axis.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=6,
            color="black" if abs(value) < 0.6 else "white",
        )
fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
axis.set_title("Corrélations de Pearson (variables numériques)")
fig.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Deux features corrélées à > 0.9 n'apportent presque rien ensemble : en garder une simplifie le modèle et son explication.
- Les arbres sont robustes à la colinéarité ; les modèles linéaires/régularisés voient leurs coefficients devenir instables.
- La corrélation ne capture pas les relations **non linéaires** : la vérifier par des graphes cible vs feature.

In [ ]:
# Outliers : comptage par la règle de l'IQR (1.5 x écart interquartile).
rows = []
for column in numeric_columns:
    series = raw[column].dropna()
    if series.empty:
        continue
    low, high = series.quantile([0.25, 0.75])
    iqr = high - low
    outliers = int(((series < low - 1.5 * iqr) | (series > high + 1.5 * iqr)).sum())
    rows.append(
        {"colonne": column, "outliers_iqr": outliers, "part": outliers / max(len(series), 1)}
    )
outlier_frame = pd.DataFrame(rows).sort_values("outliers_iqr", ascending=False)
outlier_frame.head(8).round(4)

**Ce qu'il faut retenir**

- La règle IQR **signale**, elle ne tranche pas : un outlier peut être un client légitime (grand compte, pic saisonnier).
- Le winsorising (clip aux quantiles 1-99 %) conserve les lignes et les labels, contrairement à la suppression.

In [ ]:
# Intégrité : unicité de la clé et doublons complets.
key = CONFIG.data.id_column
duplicates = int(raw.duplicated().sum())
key_duplicates = int(raw[key].duplicated().sum()) if key and key in raw.columns else 0
print(f"doublons complets            : {duplicates}")
print(f"doublons sur la clé '{key}' : {key_duplicates}")
unique_keys = raw[key].nunique() if key in raw.columns else "n/a"
print(f"identifiants uniques         : {unique_keys} / {len(raw)}")

## 7. Synthèse de l'exploration

**Lectures clés de ce jeu de données**

- La cible est **log-normale** : travailler en log-prix (ou transformer la cible) stabilise la variance et rend les erreurs comparables entre un studio et une grande maison.
- Le quartier est le premier facteur de prix (écart d'un facteur ~2 entre centre historique et périphérie nord) : c'est aussi un proxy socio-économique, à documenter sous l'angle équité.
- L'effet de la surface est **concave** : le prix au m² décroît avec la taille. Un modèle linéaire en surface sur-évalue systématiquement les grands biens et sous-évalue les studios.
- `rooms` et `property_tax_eur` sont quasi redondants avec `surface_m2` : colinéarité attendue, pénalisante pour un modèle linéaire régularisé, neutre pour les arbres.
- Le DPE pèse de plus en plus : une classe F ou G entraîne une décote explicite (interdiction progressive de location des passoires thermiques), non linéaire entre E et G.
- L'étage ne vaut que **si** l'immeuble dispose d'un ascenseur : l'interaction étage x ascenseur change le signe de l'effet (un 6e sans ascenseur est décoté, un 6e avec ascenseur est primé).
- L'année de construction a un effet en U : l'haussmannien et le très récent sont primés, les constructions des années 1960-1975 sont décotées.
- `transport_walk_min` a un effet décroissant : chaque minute compte beaucoup jusqu'à 10 minutes, presque plus au-delà de 25.
- Les erreurs sont **hétéroscédastiques** : l'erreur absolue croît avec le prix. Piloter uniquement la RMSE conduit à sacrifier les petits biens ; le MAPE et l'erreur médiane sont plus justes.
- `recent_sales_1km` n'explique pas le prix mais la **fiabilité** de l'estimation : un marché fin (< 10 ventes) doit produire une fourchette plus large, pas un prix plus précis.
- Quelques biens d'exception existent légitimement (> 1,2 M EUR) : les supprimer appauvrirait le modèle, le winsorising (0,5-99,5 %) et une analyse d'erreur dédiée sont préférables.
- `condition_score` et `condo_fees_eur` comportent des manquants volontaires (6 % et 4 %) : l'imputation doit être explicite, et un indicateur de manquant est souvent rentable.
- Le bruit injecté (multiplicateur log-normal d'environ 7 %) fixe un plafond : un MAPE proche de 0 signerait une fuite de données, pas un bon modèle. Même un modèle parfait ne couvre que ~82 % des biens à ±10 %.

### Décisions de modélisation issues de l'EDA

| Observation | Décision |
| --- | --- |
| Valeurs manquantes localisées | Imputation apprise sur le train (notebook 03) |
| Échelles hétérogènes | Scaling numérique obligatoire |
| Outliers légitimes | Winsorising plutôt que suppression |
| Modalités rares | Regroupement `rare` avant encodage |
| Colinéarité | Surveiller l'importance des features (notebook 04) |

**Suite** : `02_validation.ipynb` transforme ces observations en **contrats exécutables** (Pandera).